# MERFISH Cell-Type Classification — Solution

**UR Biomedical Data Science Hackathon, Summer 2026**

Predict one of 60 cell types for each of 5,000 test cells from 200 MERFISH gene
counts plus cell metadata. Scored on overall accuracy.

## Result

| | accuracy |
|---|---|
| Cross-validated (out-of-fold) | **0.7764** |
| Leaderboard | **0.76** |
| Majority-class baseline | 0.141 |

## Approach in one paragraph

Three measurements drove every design choice. First, the signal is split roughly evenly
between genes and metadata (genes alone 0.53, metadata alone 0.44, together 0.75) — so
metadata is a first-class input, not a garnish. Second, whether `Region` is missing
**perfectly partitions** the 60 cell types into 21 glial and 39 neuronal types with zero
overlap, which justifies training two specialist models. Third, `Segment` (combined with
`Excitatory_vs_Inhibitory`) narrows a cell to ~1.9 candidate types with leave-one-out
coverage of exactly 1.0000, so it can be applied as a **hard constraint** rather than a
soft feature.

On top of that: spatial "niche" features pooled over train+test together (label-free, so
transductive but legitimate), label-derived features built strictly out-of-fold, seven base
models per pool, and greedy ensemble selection.

## Runtime

End-to-end ~45 minutes on 10 CPU cores. The base-model fitting dominates.

## 1. Setup

In [ ]:
# pip install numpy pandas scikit-learn scipy xgboost lightgbm
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, time
from scipy.spatial import cKDTree
from scipy.special import gammaln
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder, normalize
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import ExtraTreesClassifier
import xgboost as xgb, lightgbm as lgb

SEED = 0
DATA = "data"          # folder holding counts_/meta_ train and test CSVs
OUT  = "prediction/prediction.csv"

## 2. Load the data

`index_col=0` makes the Cell ID the row label. The counts table and the meta table are two
descriptions of the same cells, so their row order must line up exactly — a silent
misalignment here would attach the wrong label to every cell.

In [ ]:
counts_train = pd.read_csv(f"{DATA}/counts_train.csv", index_col=0)
counts_test  = pd.read_csv(f"{DATA}/counts_test.csv",  index_col=0)
meta_train   = pd.read_csv(f"{DATA}/meta_train.csv",   index_col=0)
meta_test    = pd.read_csv(f"{DATA}/meta_test.csv",    index_col=0)

assert (counts_train.index == meta_train.index).all()
assert (counts_test.index  == meta_test.index).all()
assert list(counts_train.columns) == list(counts_test.columns)
assert len(set(counts_train.index) & set(counts_test.index)) == 0

Ctr = counts_train.values.astype(np.float64)
Cte = counts_test.values.astype(np.float64)
y_all = meta_train["MERFISH_cell_type_annotation"].values

# combined view: used ONLY for label-free spatial features
Call = np.vstack([Ctr, Cte])
mall = pd.concat([meta_train, meta_test])
NTR  = len(meta_train)

print("train", Ctr.shape, " test", Cte.shape, " classes", len(np.unique(y_all)))
print("median transcripts per cell:", np.median(Ctr.sum(1)))
print("zero fraction: %.3f" % (Ctr == 0).mean())

## 3. The two structural facts the solution is built on

These are verified, not assumed. Both are checked leave-one-out so the numbers reflect what
the constraint would do to an unseen cell.

In [ ]:
na = meta_train["Region"].isna()
print("cell types when Region is NaN  :", pd.unique(y_all[na.values]).size)
print("cell types when Region is known:", pd.unique(y_all[~na.values]).size)
print("overlap between the two pools  :",
      len(set(y_all[na.values]) & set(y_all[~na.values])), "<- zero means a perfect partition")

def loo_constraint(keys, subset):
    """Leave-one-out coverage and candidate-set size for a metadata grouping."""
    s, yy = meta_train[subset], pd.Series(y_all, index=meta_train.index)[subset]
    k = s[keys].astype(str).agg("|".join, axis=1)
    cov, sz = [], []
    for _, grp in yy.groupby(k):
        vc = grp.value_counts()
        for lab in grp:
            v = vc.copy(); v[lab] -= 1; v = v[v > 0]
            cov.append(lab in v.index); sz.append(len(v))
    return np.mean(cov), np.mean(sz)

for keys in (["Segment"], ["Segment", "Excitatory_vs_Inhibitory"]):
    c, s = loo_constraint(keys, ~na)
    print(f"{'+'.join(keys):<40} LOO coverage {c:.4f}  avg candidates {s:.2f}")

## 4. Feature blocks

**Gene block** — log counts, log relative abundance, library size, and how many genes were
detected. Cells differ in total counts for technical reasons, so both raw and
depth-normalised views are provided.

**Niche pooling** — sums transcripts over the *k* nearest cells in the same tissue section,
using train and test together. This touches no labels, so it is legitimate transductive
information, and it partly compensates for the very shallow ~21 transcripts per cell.

In [ ]:
def gene_block(C):
    lib = C.sum(1, keepdims=True)
    return np.hstack([np.log1p(C),
                      np.log1p(C / lib.clip(1) * 100.0),
                      np.log1p(lib),
                      (C > 0).sum(1, keepdims=True)])

def numeric_meta(meta):
    return np.hstack([
        np.log1p(meta["volume"].values)[:, None],
        meta[["center_x", "center_y"]].values / 1000.0,
        meta[["AP_position"]].values.astype(float),
        np.column_stack([meta[c].isna().values.astype(float)
                         for c in ["Region", "Segment", "Excitatory_vs_Inhibitory"]]),
    ])

def neighbour_pool(C, meta, k=10):
    """Sum of counts over the k nearest cells within the same tissue section."""
    out = np.zeros_like(C, dtype=np.float64)
    xy, sec = meta[["center_x", "center_y"]].values, meta["Section_ID"].values
    for s in np.unique(sec):
        idx = np.where(sec == s)[0]
        P = xy[idx]
        kk = min(k + 1, len(idx))
        _, nb = cKDTree(P).query(P, k=kk)
        nb = np.atleast_2d(nb)[:, 1:]          # drop self
        out[idx] = C[idx[nb]].sum(1)
    return out

## 5. Label-derived features

Each of these summarises the labels of a *reference* set of cells. The caller always passes a
reference that excludes the cells the features are attached to, which is what keeps the
cross-validation honest.

- `group_priors` — smoothed cell-type distribution within each metadata group (section, segment, mouse, ...)
- `spatial_label_dist` — distance-weighted label distribution of nearby cells in the same section
- `expr_label_dist` — label distribution of the nearest cells in expression space
- `nb_loglik` — multinomial naive-Bayes class log-likelihood, a generative summary of the gene evidence

In [ ]:
GROUP_COLS = ["Section_ID", "Segment", "Region", "Datasets", "Mouse_ID",
              "AP_position", "Excitatory_vs_Inhibitory", "Gender"]

def _onehot(yi, NC):
    Y = np.zeros((len(yi), NC)); Y[np.arange(len(yi)), yi] = 1.0
    return Y

def group_priors(m_ref, yi_ref, m_q, NC, cols=GROUP_COLS, alpha=5.0):
    gp = np.bincount(yi_ref, minlength=NC).astype(float); gp /= gp.sum()
    out = []
    for c in cols:
        rv, qv = m_ref[c].astype(str).values, m_q[c].astype(str).values
        cnt = (pd.DataFrame({"g": rv, "y": yi_ref}).groupby(["g", "y"]).size()
                 .unstack(fill_value=0).reindex(columns=range(NC), fill_value=0))
        P = (cnt.values + alpha * gp[None, :]) / (cnt.sum(1).values[:, None] + alpha)
        idx = {g: i for i, g in enumerate(cnt.index)}
        Q = np.tile(gp, (len(qv), 1))
        hit = np.array([idx.get(v, -1) for v in qv]); m = hit >= 0
        Q[m] = P[hit[m]]
        out.append(Q)
    return np.hstack(out)

def spatial_label_dist(m_ref, yi_ref, m_q, NC, ks=(5, 15, 40), alpha=3.0):
    gp = np.bincount(yi_ref, minlength=NC).astype(float); gp /= gp.sum()
    Yr = _onehot(yi_ref, NC)
    xq, xr = m_q[["center_x","center_y"]].values, m_ref[["center_x","center_y"]].values
    sq, sr = m_q["Section_ID"].values, m_ref["Section_ID"].values
    outs = [np.tile(gp, (len(m_q), 1)) for _ in ks]
    dist = np.full((len(m_q), len(ks)), 500.0)
    for s in np.unique(sq):
        qi, ri = np.where(sq == s)[0], np.where(sr == s)[0]
        if len(ri) == 0: continue
        kmax = min(max(ks), len(ri))
        d, j = cKDTree(xr[ri]).query(xq[qi], k=kmax)
        d = np.atleast_2d(d.reshape(len(qi), kmax)); j = np.atleast_2d(j.reshape(len(qi), kmax))
        for a, k in enumerate(ks):
            kk = min(k, j.shape[1])
            w = 1.0 / (d[:, :kk] + 50.0)
            acc = (Yr[ri[j[:, :kk]]] * w[:, :, None]).sum(1)
            outs[a][qi] = (acc + alpha * gp[None, :]) / (w.sum(1)[:, None] + alpha)
            dist[qi, a] = d[:, kk - 1]
    return np.hstack(outs + [np.log1p(dist)])

def expr_label_dist(Xr, yi_ref, Xq, NC, ks=(10, 30, 75), alpha=3.0):
    gp = np.bincount(yi_ref, minlength=NC).astype(float); gp /= gp.sum()
    Yr = _onehot(yi_ref, NC)
    S = normalize(Xq) @ normalize(Xr).T                 # cosine similarity
    kmax = min(max(ks), S.shape[1])
    part = np.argpartition(-S, kmax - 1, axis=1)[:, :kmax]
    sv = np.take_along_axis(S, part, 1)
    order = np.argsort(-sv, axis=1)
    part, sv = np.take_along_axis(part, order, 1), np.take_along_axis(sv, order, 1)
    outs = []
    for k in ks:
        kk = min(k, kmax)
        w = np.clip(sv[:, :kk], 0, None) ** 3
        acc = (Yr[part[:, :kk]] * w[:, :, None]).sum(1)
        outs.append((acc + alpha * gp[None, :]) / (w.sum(1)[:, None] + alpha))
    return np.hstack(outs + [sv[:, :5]])

def nb_loglik(C_ref, yi_ref, C_q, NC, alpha=0.4):
    G = C_ref.shape[1]
    prof = np.zeros((NC, G))
    for c in range(NC):
        m = yi_ref == c
        if m.any(): prof[c] = C_ref[m].sum(0)
    prof = (prof + alpha) / (prof.sum(1, keepdims=True) + alpha * G)
    L = C_q @ np.log(prof).T
    return L - L.max(1, keepdims=True)

def label_features(m_ref, yi_ref, C_ref, m_q, C_q, NC):
    return np.hstack([
        group_priors(m_ref, yi_ref, m_q, NC),
        spatial_label_dist(m_ref, yi_ref, m_q, NC),
        expr_label_dist(np.log1p(C_ref), yi_ref, np.log1p(C_q), NC),
        nb_loglik(C_ref, yi_ref, C_q, NC),
    ])

## 6. Hard candidate masks

Every key below was verified leave-one-out at coverage **1.0000** — it never excludes the true
label. Together they cut the candidate set from 60 to ~10.6 on average and fully determine
555 of the 5,000 test cells.

In [ ]:
HARD = [("__nanflag__", 1), ("Segment", 3), ("Region", 3), ("Segment|ExVsIn", 3)]

def _mask_keys(meta):
    seg = meta["Segment"].astype(str).to_numpy(dtype="U32")
    exi = meta["Excitatory_vs_Inhibitory"].astype(str).to_numpy(dtype="U32")
    return {"Segment": seg,
            "Region": meta["Region"].astype(str).to_numpy(dtype="U32"),
            "Segment|ExVsIn": np.char.add(np.char.add(seg, "|"), exi),
            "__nanflag__": meta["Region"].isna().to_numpy().astype(int).astype("U32")}

def build_masks(m_ref, yi_ref, m_q, NC):
    kr, kq = _mask_keys(m_ref), _mask_keys(m_q)
    M = np.ones((len(m_q), NC), dtype=bool)
    for col, mn in HARD:
        rv, qv = kr[col], kq[col]
        for g in np.unique(rv):
            sel = rv == g
            if sel.sum() < mn: continue
            v = np.zeros(NC, dtype=bool); v[np.unique(yi_ref[sel])] = True
            rows = qv == g
            if rows.any(): M[rows] &= v
    M[~M.any(1)] = True                      # never leave a row with no options
    return M

def apply_mask(P, M):
    Q = P * M
    s = Q.sum(1, keepdims=True)
    bad = s[:, 0] <= 1e-12
    Q[bad] = P[bad]
    return Q / np.clip(Q.sum(1, keepdims=True), 1e-12, None)

## 7. Base models

Seven per pool. The notable one is `DirMultNB`: a Dirichlet-multinomial naive Bayes. Plain
multinomial NB assumes counts are multinomial draws, but real MERFISH counts are
overdispersed relative to that; the Dirichlet-multinomial absorbs the extra variance.

In [ ]:
class DirMultNB:
    """Dirichlet-multinomial naive Bayes - tolerates overdispersed counts."""
    def __init__(self, conc=200.0, alpha=0.2):
        self.conc, self.alpha = conc, alpha

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        G = X.shape[1]
        P = np.zeros((len(self.classes_), G))
        for i, c in enumerate(self.classes_):
            P[i] = X[y == c].sum(0)
        P = (P + self.alpha) / (P.sum(1, keepdims=True) + self.alpha * G)
        self.A = P * self.conc
        cnt = np.array([(y == c).sum() for c in self.classes_], float)
        self.logprior = np.log(cnt / cnt.sum() + 1e-12)
        return self

    def predict_proba(self, X):
        A, n = self.A, X.sum(1)
        a0 = A.sum(1)
        t = (gammaln(X[:, None, :] + A[None, :, :]) - gammaln(A[None, :, :])).sum(2)
        L = gammaln(a0)[None, :] - gammaln(n[:, None] + a0[None, :]) + t + self.logprior
        L -= L.max(1, keepdims=True)
        e = np.exp(L)
        return e / e.sum(1, keepdims=True)

def base_models():
    return {
        "dm":  lambda: DirMultNB(conc=200.0),
        "lr":  lambda: make_pipeline(StandardScaler(),
                                     LogisticRegression(max_iter=4000, C=0.5, n_jobs=-1)),
        "mlp": lambda: MLPClassifier(hidden_layer_sizes=(256,128), alpha=1e-3, max_iter=500,
                                     early_stopping=True, random_state=SEED),
        "xgb": lambda: xgb.XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.06,
                                         subsample=0.8, colsample_bytree=0.5,
                                         tree_method="hist", n_jobs=10, verbosity=0,
                                         random_state=SEED),
        "xgb2":lambda: xgb.XGBClassifier(n_estimators=700, max_depth=9, learning_rate=0.035,
                                         subsample=0.75, colsample_bytree=0.35,
                                         min_child_weight=3, reg_lambda=2.0,
                                         tree_method="hist", n_jobs=10, verbosity=0,
                                         random_state=SEED+1),
        "lgb": lambda: lgb.LGBMClassifier(n_estimators=600, learning_rate=0.05, num_leaves=63,
                                          subsample=0.8, subsample_freq=1,
                                          colsample_bytree=0.35, min_child_samples=8,
                                          n_jobs=10, verbose=-1, random_state=SEED),
        "et":  lambda: ExtraTreesClassifier(n_estimators=800, n_jobs=10,
                                            random_state=SEED, min_samples_leaf=2),
    }

RAW_MODELS = {"dm"}    # consumes raw counts rather than the engineered block

## 8. The pool specialist

One `Pool` = all cells sharing a `Region`-NaN status. Because that split is exact, a glial
model never spends capacity on neuronal classes and vice versa.

Note `_oof_lab`: label-derived features for the training rows are themselves built out-of-fold,
so a cell's features never depend on its own label.

In [ ]:
class Pool:
    def __init__(self, name, sel_tr, sel_te):
        self.name = name
        self.itr, self.ite = np.where(sel_tr)[0], np.where(sel_te)[0]
        self.mtr, self.mte = meta_train.iloc[self.itr], meta_test.iloc[self.ite]
        self.Ctr, self.Cte = Ctr[self.itr], Cte[self.ite]
        self.le = LabelEncoder().fit(y_all[self.itr])
        self.y = self.le.transform(y_all[self.itr])
        self.NC = len(self.le.classes_)
        self._static()

    def _static(self):
        """Label-free features, computed over train+test together."""
        keep = np.concatenate([self.itr, NTR + self.ite])
        Cp, mp = Call[keep], mall.iloc[keep]
        parts = [gene_block(Cp), numeric_meta(mp)]
        for k in (8, 25):
            P = neighbour_pool(Cp, mp, k=k)
            parts.append(np.log1p(P / P.sum(1, keepdims=True).clip(1) * 100))
        S = np.hstack(parts)
        self.Str, self.Ste = S[:len(self.itr)], S[len(self.itr):]

    def _lab(self, ref, q_meta, q_C):
        return label_features(self.mtr.iloc[ref], self.y[ref], self.Ctr[ref],
                              q_meta, q_C, self.NC)

    def _oof_lab(self, idx, seed=1):
        out = None
        for a, b in StratifiedKFold(5, shuffle=True, random_state=seed).split(idx, self.y[idx]):
            blk = self._lab(idx[a], self.mtr.iloc[idx[b]], self.Ctr[idx[b]])
            if out is None: out = np.zeros((len(idx), blk.shape[1]))
            out[b] = blk
        return out

    def fit_cv(self):
        folds = list(StratifiedKFold(5, shuffle=True, random_state=SEED).split(self.Ctr, self.y))
        mods = base_models()
        self.oof = {k: np.zeros((len(self.y), self.NC)) for k in mods}
        self.Mcv = np.ones((len(self.y), self.NC), bool)
        for tr, te in folds:
            Xtr = np.hstack([self.Str[tr], self._oof_lab(tr)])
            Xte = np.hstack([self.Str[te], self._lab(tr, self.mtr.iloc[te], self.Ctr[te])])
            self.Mcv[te] = build_masks(self.mtr.iloc[tr], self.y[tr], self.mtr.iloc[te], self.NC)
            for k, mk in mods.items():
                if k in RAW_MODELS:
                    p = mk().fit(self.Ctr[tr], self.y[tr]).predict_proba(self.Ctr[te])
                else:
                    p = mk().fit(Xtr, self.y[tr]).predict_proba(Xte)
                self.oof[k][te] = p
        for k in mods:
            print(f"   [{self.name}] {k:<5} {(self.oof[k].argmax(1)==self.y).mean():.4f}")

    def fit_full(self):
        idx = np.arange(len(self.y))
        Xtr = np.hstack([self.Str, self._oof_lab(idx, seed=2)])
        Xte = np.hstack([self.Ste, self._lab(idx, self.mte, self.Cte)])
        self.Mte = build_masks(self.mtr, self.y, self.mte, self.NC)
        self.test_p = {}
        for k, mk in base_models().items():
            if k in RAW_MODELS:
                self.test_p[k] = mk().fit(self.Ctr, self.y).predict_proba(self.Cte)
            else:
                self.test_p[k] = mk().fit(Xtr, self.y).predict_proba(Xte)

## 9. Ensemble selection and prior calibration

Greedy hill-climbing with replacement (Caruana ensemble selection) rather than a logistic
stacker. This was not a stylistic choice: a logistic stacker on these probabilities scored
0.6381 on the glial pool against 0.7005 for XGBoost alone — it actively destroyed signal.

Weight selection is nested — chosen on 4/5 of the cells, scored on the held-out 1/5.

In [ ]:
def acc(P, y): return (P.argmax(1) == y).mean()

def blend(probs, w):
    tot = sum(w.values())
    return sum(w[k] * probs[k] for k in w if w[k]) / tot

def greedy_ensemble(oof, y, keys, n_iter=40):
    w = {k: 0 for k in keys}
    w[max(keys, key=lambda k: acc(oof[k], y))] = 1
    best = acc(blend(oof, w), y)
    for _ in range(n_iter):
        cand, ca = None, best
        for k in keys:
            w[k] += 1
            a = acc(blend(oof, w), y)
            w[k] -= 1
            if a > ca + 1e-6: cand, ca = k, a
        if cand is None: break
        w[cand] += 1; best = ca
    return w, best

def sinkhorn_prior(P, target, n_iter=200, damp=1.0):
    """Re-weight classes so predicted marginals approach `target`."""
    Q, w = P.copy(), np.ones(P.shape[1])
    for _ in range(n_iter):
        w *= ((target + 1e-9) / (Q.mean(0) + 1e-9)) ** damp
        Q = P * w
        Q /= Q.sum(1, keepdims=True).clip(1e-12)
    return Q

def global_prior_match(P, y_ref, NC, damp=0.5):
    t = np.bincount(y_ref, minlength=NC).astype(float); t /= t.sum()
    return sinkhorn_prior(P, t, damp=damp)

In [ ]:
CALIBRATION = {"glia": "global", "neuron": "none"}

t0 = time.time()
na_tr = meta_train["Region"].isna().values
na_te = meta_test["Region"].isna().values
pools = [Pool("glia", na_tr, na_te), Pool("neuron", ~na_tr, ~na_te)]

correct = 0
pred = np.empty(len(meta_test), dtype=object)

for p in pools:
    print(f"[{p.name}] {len(p.itr)} train / {len(p.ite)} test cells, {p.NC} classes")
    p.fit_cv()
    p.fit_full()
    keys = sorted(p.oof)
    oofm = {k: apply_mask(p.oof[k].copy(), p.Mcv) for k in keys}
    tem  = {k: apply_mask(p.test_p[k].copy(), p.Mte) for k in keys}

    # honest CV: weights chosen on 4/5, scored on the held-out 1/5
    P_oof = np.zeros((len(p.y), p.NC))
    for a, b in StratifiedKFold(5, shuffle=True, random_state=11).split(p.y, p.y):
        w, _ = greedy_ensemble({k: oofm[k][a] for k in keys}, p.y[a], keys)
        P_oof[b] = blend({k: oofm[k][b] for k in keys}, w)
    if CALIBRATION[p.name] == "global":
        Pc = np.zeros_like(P_oof)
        for a, b in StratifiedKFold(2, shuffle=True, random_state=13).split(p.y, p.y):
            Pc[b] = global_prior_match(P_oof[b], p.y[a], p.NC)
        P_oof = apply_mask(Pc, p.Mcv)
    a_pool = acc(P_oof, p.y)
    print(f"   [{p.name}] honest CV {a_pool:.4f}")
    correct += a_pool * len(p.y)

    # final: weights from all training cells, applied to the test probabilities
    w, _ = greedy_ensemble(oofm, p.y, keys)
    print(f"   [{p.name}] weights {dict((k,v) for k,v in w.items() if v)}")
    Pte = blend(tem, w)
    if CALIBRATION[p.name] == "global":
        Pte = apply_mask(global_prior_match(Pte, p.y, p.NC), p.Mte)
    pred[p.ite] = p.le.inverse_transform(Pte.argmax(1))

print(f"\n=== OVERALL CV ACCURACY: {correct/len(meta_train):.4f} ===  ({time.time()-t0:.0f}s)")

## 11. Write and validate the submission

In [ ]:
out = pd.DataFrame({"Cell_ID": meta_test.index,
                    "MERFISH_cell_type_annotation.y": pred})
out.to_csv(OUT, index=False)

# format contract: 5000 rows, meta_test order, labels drawn from the 60 training types
assert len(out) == 5000
assert (out["Cell_ID"].astype(str).values == meta_test.index.astype(str).values).all()
assert set(out["MERFISH_cell_type_annotation.y"]) <= set(y_all)
assert out.isna().sum().sum() == 0
print("wrote", OUT, out.shape)
out.head()